In [1]:
import os
import torch
import torch.nn
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np

cwd = Path(os.getcwd())
images_lib = cwd / "data" / "images" / "Images"
annotations_lib = cwd / "data" / "Annotation"

dtype = torch.float
device = torch.device("cuda:0")

In [2]:
class labeled_image:
    def __init__(self, image_path, meta_path):
        self.image = Image.open(str(image_path)+".jpg")
        self.tensor = torch.tensor(np.array(Image.open(str(image_path)+".jpg")), device=device, dtype=dtype)
        self.meta = {node.tag: node.text for node in ET.parse(str(meta_path)).getroot().iter() }
        [self.meta.pop(key, None) for key in dict(self.meta) if '\n' in self.meta[key]]
        
def batch_generator(file_list, batch_size):
    n_total = len(file_list)
    n_images_left= len(file_list)
    n_images_used = 0
    assert n_images_left%batch_size == 0
    def _batch_generator():
        nonlocal n_total
        nonlocal n_images_left
        nonlocal n_images_used
        nonlocal batch_size
        if n_images_left == 0:
            return []
        else:
            n_images_left -= batch_size
            return_list = [labeled_image(file_list[n][0], file_list[n][1]) for n in range(n_images_used, n_images_used+batch_size)]
            n_images_used += batch_size
            return return_list
    return _batch_generator

In [3]:
file_list = []
for directory in os.listdir(annotations_lib):
    for file in os.listdir(annotations_lib / directory):
        file_list += [(str(images_lib / directory / file), str(annotations_lib / directory / file))]

In [4]:
get_batch = batch_generator(file_list, 3)

In [5]:
input_data = get_batch()